# SAE Single-Feature Steering: Interpretability

Для каждой из **top-K отдельных SAE-фичей** запускаем steering на N вопросах и смотрим,
как именно **одна фича** меняет ответ модели.

Это позволяет:
- Понять, **что делает** каждая фича (хеджирование? отказ? смена темы?)
- Выбрать фичи для финального delta-вектора осознанно
- Отсечь «мусорные» фичи, которые ломают генерацию

**Загрузите:**
1. `test.csv` — вопросы с `question`, `verbal_uncertainty`, `sentence_semantic_entropy`
2. `sae_feature_analysis_merged_stats.json` — результат robust analysis

## 0. GPU

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"Python: {sys.version}")

## 1. Клон репозитория и зависимости

In [ ]:
import os, sys, subprocess

GIT_URL = "https://github.com/SadreevAmir/sae-muc.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/sae-muc"

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("-> Installing dependencies ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "numpy>=2.0.0,<2.1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "--no-cache-dir", "transformers>=4.40", "accelerate"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub"])

import torch
assert torch.cuda.is_available(), "GPU not available!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Загрузка файлов

In [ ]:
import os

UPLOAD_FILES = True  # @param {type:"boolean"}
UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

_uploaded_csv = None
_uploaded_json = None

if UPLOAD_FILES:
    from google.colab import files

    print("Step 1/2: Upload test.csv")
    up1 = files.upload()
    for fn, data in up1.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".csv"):
            _uploaded_csv = dst

    print("\nStep 2/2: Upload stats JSON")
    up2 = files.upload()
    for fn, data in up2.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".json"):
            _uploaded_json = dst

    print(f"\nCSV:  {_uploaded_csv}")
    print(f"JSON: {_uploaded_json}")
else:
    print("Set paths manually in section 3.")

## 3. Конфигурация

| Параметр | Описание |
|---|---|
| `N_QUESTIONS` | Сколько вопросов прогнать через каждую фичу |
| `TOP_N_FEATURES` | Сколько отдельных фичей тестировать (top по mean-diff) |
| `ALPHA` | Фиксированная сила вмешательства |
| `TARGET_LAYER` | Слой для анализа (None = все слои из stats) |

In [ ]:
TEST_CSV    = _uploaded_csv  or "/content/uploads/test.csv"        # @param {type:"string"}
STATS_JSON  = _uploaded_json or "/content/uploads/sae_feature_analysis_merged_stats.json"  # @param {type:"string"}

MODEL_NAME  = "Mistral-7B-Instruct-v0.3"  # @param {type:"string"}
SAE_RELEASE = "mistral-7b-res-wg"          # @param {type:"string"}
SAE_DTYPE   = "float32"                    # @param ["float32", "float16", "bfloat16"]

N_QUESTIONS     = 10   # @param {type:"integer"}
TOP_N_FEATURES  = 10   # @param {type:"integer"}
ALPHA           = 10.0 # @param {type:"number"}
TARGET_LAYER    = None # @param {type:"raw"}  (None = all layers, or e.g. 23)

OUTPUT_DIR = "/content/single_feature_results"  # @param {type:"string"}

import os, json
import pandas as pd

assert os.path.isfile(TEST_CSV),   f"CSV not found: {TEST_CSV}"
assert os.path.isfile(STATS_JSON), f"JSON not found: {STATS_JSON}"

df_all = pd.read_csv(TEST_CSV)
print(f"CSV: {len(df_all)} rows")
print(f"Will use {N_QUESTIONS} questions x {TOP_N_FEATURES} features per layer")
print(f"Alpha = {ALPHA}")
df_all.head(3)

## 4. HuggingFace Login

In [ ]:
from huggingface_hub import login
login()

## 5. Загрузка модели, SAE, выбор фичей и вопросов

In [ ]:
import json, time
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
from sae_lens import SAE

import sys, os
REPO_DIR = "/content/sae-muc"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from sae_muc.hooks import register_sae_latent_hooks, clear_sae_latent_hooks
from sae_muc.layer_map import hf_layers_for_release
from sae_muc.prompts_mini import make_sentence_user_content

torch.manual_seed(42)
np.random.seed(42)

# ── Resolve model name ──
if "Mistral" in MODEL_NAME:
    full_model_name = f"mistralai/{MODEL_NAME}"
elif "Llama" in MODEL_NAME:
    full_model_name = f"meta-llama/{MODEL_NAME}"
elif "Qwen" in MODEL_NAME:
    full_model_name = f"Qwen/{MODEL_NAME}"
else:
    full_model_name = MODEL_NAME

# ── Parse stats JSON: extract per-layer top features ──
with open(STATS_JSON, "r") as f:
    stats = json.load(f)

expected_layers = {l for l, _ in hf_layers_for_release(SAE_RELEASE)}

layer_features_map = OrderedDict()

for item in stats.get("layers", []):
    layer = int(item["layer"])
    if layer not in expected_layers:
        continue
    if TARGET_LAYER is not None and layer != TARGET_LAYER:
        continue

    sae_id = str(item["sae_id"])
    feat_list = item.get("top_uncertainty_feature_idx", [])[:TOP_N_FEATURES]

    bootstrap_stable = set(item.get("bootstrap", {}).get("stable_uncertainty_up_top50", []))
    perm_sig = set(item.get("permutation", {}).get("candidate_perm_significant_005", []))
    perm_pvals = item.get("permutation", {}).get("candidate_perm_pvalues", {})

    ttest_unc_up = set(item.get("ttest", {}).get("uncertainty_up_top50_by_abs_t", []))

    cohens_d_unc = item.get("cohens_d", {}).get("top_uncertainty_up_by_abs_d", [])
    cohens_d_rank = {int(fi): rank for rank, fi in enumerate(cohens_d_unc)}

    features = []
    for rank, fi in enumerate(feat_list):
        pval = perm_pvals.get(str(fi), None)
        features.append({
            "feature_idx": int(fi),
            "rank": rank,
            "perm_pvalue": pval,
            "bootstrap_stable": fi in bootstrap_stable,
            "perm_significant": fi in perm_sig,
            "ttest_significant": fi in ttest_unc_up,
            "cohens_d_rank": cohens_d_rank.get(fi, None),
        })

    layer_features_map[layer] = {"sae_id": sae_id, "features": features}

# ── Display selected features ──
print(f"Layers: {list(layer_features_map.keys())}")
print(f"Features per layer: {TOP_N_FEATURES}")
print(f"(rank = position in top_uncertainty_feature_idx, sorted by mean_diff)\n")

for layer, info in layer_features_map.items():
    print(f"Layer {layer} ({info['sae_id']}):")
    print(f"  {'rank':<5} {'feat_idx':<10} {'perm_p':>8} {'boot':>6} {'perm<.05':>9} {'ttest':>6} {'d_rank':>7}")
    print(f"  {'-'*56}")
    for ft in info["features"]:
        pp = f"{ft['perm_pvalue']:.4f}" if ft['perm_pvalue'] is not None else '-'
        bs = 'Y' if ft['bootstrap_stable'] else '-'
        pm = 'Y' if ft['perm_significant'] else '-'
        tt = 'Y' if ft['ttest_significant'] else '-'
        dr = str(ft['cohens_d_rank']) if ft['cohens_d_rank'] is not None else '-'
        print(f"  {ft['rank']:<5} {ft['feature_idx']:<10} {pp:>8} {bs:>6} {pm:>9} {tt:>6} {dr:>7}")
    print()

# ── Select questions (mix of confident and uncertain) ──
df_sorted = df_all.sort_values("verbal_uncertainty")
n_half = N_QUESTIONS // 2
df_sel = pd.concat([
    df_sorted.head(n_half),
    df_sorted.tail(N_QUESTIONS - n_half),
]).reset_index(drop=True)

questions = df_sel["question"].astype(str).tolist()
messages = [[{"role": "user", "content": make_sentence_user_content(q)}] for q in questions]
print(f"Selected {len(questions)} questions ({n_half} confident + {N_QUESTIONS - n_half} uncertain)")

# ── Load SAEs ──
print("\nLoading SAEs ...")
layer_to_sae = {}
for layer, info in layer_features_map.items():
    t0 = time.time()
    sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=info["sae_id"], device="cpu", dtype=SAE_DTYPE)
    layer_to_sae[layer] = sae
    print(f"  Layer {layer}: d_sae={sae.cfg.d_sae}, loaded in {time.time()-t0:.1f}s")

# ── Load LLM ──
print(f"\nLoading {full_model_name} ...")
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(full_model_name, torch_dtype=torch.float16, device_map="auto")
model.eval()
tokenizer = AutoTokenizer.from_pretrained(full_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Model loaded in {time.time()-t0:.1f}s")

## 6. Генерация: baseline + каждая фича по отдельности

Для каждого слоя:
1. Генерируем **baseline** (без хуков)
2. Для каждой из top-N фичей — генерируем со steering через **одну** фичу

Результаты сохраняются и выводятся прямо в ноутбук.

In [ ]:
from tqdm.auto import tqdm
from IPython.display import display, HTML

os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_greedy(model, tokenizer, batch_messages):
    inputs = tokenizer.apply_chat_template(
        batch_messages, tokenize=True, add_generation_prompt=True,
        truncation=True, padding=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return tokenizer.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

all_results = {}  # (layer, feature_idx_or_'baseline') -> list of answers

for layer, info in layer_features_map.items():
    sae = layer_to_sae[layer]
    d_sae = sae.cfg.d_sae

    # ── Baseline ──
    print(f"\n{'='*80}")
    print(f"Layer {layer}: generating BASELINE ({len(questions)} questions) ...")
    clear_sae_latent_hooks(model)
    baseline_answers = generate_greedy(model, tokenizer, messages)
    all_results[(layer, "baseline")] = baseline_answers

    # ── Per-feature steering ──
    for ft in tqdm(info["features"], desc=f"Layer {layer} features"):
        fi = ft["feature_idx"]

        delta = torch.zeros(d_sae, dtype=torch.float32)
        delta[fi] = 1.0
        delta = delta / (delta.norm() + 1e-8)

        single_sae = {layer: sae}
        single_delta = {layer: delta}

        clear_sae_latent_hooks(model)
        register_sae_latent_hooks(model, single_sae, single_delta, [layer], ALPHA)

        steered_answers = generate_greedy(model, tokenizer, messages)
        all_results[(layer, fi)] = steered_answers

    clear_sae_latent_hooks(model)

print(f"\nDone: {len(all_results)} runs x {len(questions)} questions")

## 7. Результаты: таблица сравнения

Для каждого слоя выводим таблицу: **baseline** vs **каждая фича**.

In [ ]:
import html as html_mod
from IPython.display import display, HTML

for layer, info in layer_features_map.items():
    baseline = all_results[(layer, "baseline")]

    display(HTML(f"<h2>Layer {layer} ({info['sae_id']}), alpha={ALPHA}</h2>"))

    for qi, q in enumerate(questions):
        rows_html = ""
        base_ans = baseline[qi].strip()[:200]

        rows_html += f"""
        <tr style='background:#f0f0f0'>
            <td><b>baseline</b></td><td>-</td><td>-</td><td>-</td><td>-</td>
            <td>{html_mod.escape(base_ans)}</td>
            <td>-</td>
        </tr>"""

        for ft in info["features"]:
            fi = ft["feature_idx"]
            ans = all_results[(layer, fi)][qi].strip()[:200]
            changed = ans != base_ans
            bg = '#fff3cd' if changed else '#ffffff'
            pp = f"{ft['perm_pvalue']:.4f}" if ft['perm_pvalue'] is not None else '-'
            bs = 'Y' if ft['bootstrap_stable'] else '-'
            pm = 'Y' if ft['perm_significant'] else '-'
            dr = str(ft['cohens_d_rank']) if ft['cohens_d_rank'] is not None else '-'

            rows_html += f"""
            <tr style='background:{bg}'>
                <td><b>{fi}</b></td><td>{pp}</td><td>{bs}</td><td>{pm}</td><td>{dr}</td>
                <td>{html_mod.escape(ans)}</td>
                <td>{'<b>CHANGED</b>' if changed else 'same'}</td>
            </tr>"""

        display(HTML(f"""
        <details open>
        <summary style='font-size:14px; cursor:pointer; margin:8px 0'>
            <b>Q{qi+1}:</b> {html_mod.escape(q[:100])}
        </summary>
        <table border='1' cellpadding='4' cellspacing='0' style='border-collapse:collapse; font-size:12px; width:100%'>
        <tr><th>Feature</th><th>perm_p</th><th>boot</th><th>perm&lt;.05</th><th>d_rank</th><th>Answer (first 200 chars)</th><th>Status</th></tr>
        {rows_html}
        </table>
        </details>
        """))

## 8. Сводная статистика: какие фичи реально меняют ответ

In [ ]:
import pandas as pd
from IPython.display import display

summary_rows = []

for layer, info in layer_features_map.items():
    baseline = all_results[(layer, "baseline")]

    for ft in info["features"]:
        fi = ft["feature_idx"]
        steered = all_results[(layer, fi)]

        n_changed = sum(1 for b, s in zip(baseline, steered) if b.strip() != s.strip())
        avg_len_base = np.mean([len(a) for a in baseline])
        avg_len_steer = np.mean([len(a) for a in steered])

        summary_rows.append({
            "layer": layer,
            "feature_idx": fi,
            "rank": ft["rank"],
            "perm_p": f"{ft['perm_pvalue']:.4f}" if ft['perm_pvalue'] is not None else "-",
            "bootstrap": ft["bootstrap_stable"],
            "perm_sig": ft["perm_significant"],
            "ttest_sig": ft["ttest_significant"],
            "d_rank": ft["cohens_d_rank"],
            "changed": n_changed,
            "changed_pct": f"{100*n_changed/len(questions):.0f}%",
            "avg_len_base": f"{avg_len_base:.0f}",
            "avg_len_steer": f"{avg_len_steer:.0f}",
        })

df_summary = pd.DataFrame(summary_rows)
print(f"Alpha = {ALPHA}, {len(questions)} questions per feature\n")
display(df_summary)

## 9. Сохранение в JSON и скачивание

In [ ]:
import json
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for layer, info in layer_features_map.items():
    baseline = all_results[(layer, "baseline")]

    layer_data = {
        "layer": layer,
        "sae_id": info["sae_id"],
        "alpha": ALPHA,
        "n_questions": len(questions),
        "questions": questions,
        "baseline_answers": baseline,
        "features": [],
    }

    for ft in info["features"]:
        fi = ft["feature_idx"]
        steered = all_results[(layer, fi)]
        layer_data["features"].append({
            "feature_idx": fi,
            "rank": ft["rank"],
            "perm_pvalue": ft["perm_pvalue"],
            "bootstrap_stable": ft["bootstrap_stable"],
            "perm_significant": ft["perm_significant"],
            "ttest_significant": ft["ttest_significant"],
            "cohens_d_rank": ft["cohens_d_rank"],
            "steered_answers": steered,
        })

    out_path = out_dir / f"layer_{layer}_single_features.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(layer_data, f, indent=2, ensure_ascii=False)
    print(f"Saved: {out_path}")

csv_path = out_dir / "feature_impact_summary.csv"
df_summary.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

try:
    from google.colab import files
    for p in out_dir.glob("*"):
        files.download(str(p))
    print("\nDownloads started.")
except ImportError:
    print(f"Files in: {out_dir}")